Env setup and import

In [2]:
import torch
from torch import nn
from env_simplified import make_env
from torchrl.modules import MultiAgentConvNet, MultiAgentMLP, ProbabilisticActor, MaskedCategorical
from tensordict.nn import TensorDictModule
from torchrl.collectors import MultiSyncCollector, Collector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.objectives import ClipPPOLoss, ValueEstimators

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
env = make_env()

c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-25 15:17:41,866	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


cuda


c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\torchrl\envs\libs\pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(


New policy



In [3]:
import torch
from torch import nn, Tensor
from typing import *
from functools import partial

# ---------------------------------------------------------------------
# 1. Core building blocks (unchanged)
# ---------------------------------------------------------------------

class ChannelAttention(nn.Module):
    def __init__(self, channels, ratio=16, actv_builder=nn.ReLU, bias=True):
        super().__init__()
        self.shared_mlp = nn.Sequential(
            nn.Linear(channels, channels // ratio, bias=bias),
            actv_builder(),
            nn.Linear(channels // ratio, channels, bias=bias),
        )
        if bias:
            for mod in self.modules():
                if isinstance(mod, nn.Linear):
                    nn.init.constant_(mod.bias, 0)

    def forward(self, x: Tensor):
        avg_out = self.shared_mlp(x.mean(-1))
        max_out = self.shared_mlp(x.amax(-1))
        weight = (avg_out + max_out).sigmoid()
        x = weight.unsqueeze(-1) * x
        return x


class ResBlock(nn.Module):
    def __init__(
        self,
        channels,
        dilation=1,  # <-- NEW: dilation factor
        *,
        norm_builder=nn.Identity,
        actv_builder=nn.ReLU,
        pre_actv=False,
    ):
        super().__init__()
        self.pre_actv = pre_actv

        # Padding must equal dilation to keep the sequence length unchanged
        pad = dilation 

        if pre_actv:
            self.res_unit = nn.Sequential(
                norm_builder(),
                actv_builder(),
                nn.Conv1d(channels, channels, kernel_size=3, padding=pad, dilation=dilation, bias=False),
                norm_builder(),
                actv_builder(),
                nn.Conv1d(channels, channels, kernel_size=3, padding=pad, dilation=dilation, bias=False),
            )
        else:
            self.res_unit = nn.Sequential(
                nn.Conv1d(channels, channels, kernel_size=3, padding=pad, dilation=dilation, bias=False),
                norm_builder(),
                actv_builder(),
                nn.Conv1d(channels, channels, kernel_size=3, padding=pad, dilation=dilation, bias=False),
                norm_builder(),
            )
            self.actv = actv_builder()
        self.ca = ChannelAttention(channels, actv_builder=actv_builder, bias=True)

    # forward() remains identical
    def forward(self, x):
        out = self.res_unit(x)
        out = self.ca(out)
        out = out + x
        if not self.pre_actv:
            out = self.actv(out)
        return out


class ResNet(nn.Module):
    def __init__(
        self,
        in_channels,
        conv_channels,
        num_blocks,
        seq_len,
        *,
        actv_builder=nn.Mish,
        pre_actv=True,
    ):
        super().__init__()

        # FIX: explicitly use conv_channels here
        norm_builder = partial(nn.BatchNorm1d, conv_channels, momentum=0.01, eps=1e-3)

        blocks = []
        for i in range(num_blocks):
            dilation = 2 ** i
            blocks.append(ResBlock(
                conv_channels,
                dilation=dilation,
                norm_builder=norm_builder, # type: ignore
                actv_builder=actv_builder, # type: ignore
                pre_actv=pre_actv,
            ))

        layers = [nn.Conv1d(in_channels, conv_channels, kernel_size=3, padding=1, bias=False)]
        if pre_actv:
            layers += [*blocks, norm_builder(), actv_builder()]
        else:
            layers += [norm_builder(), actv_builder(), *blocks]
        layers += [
            nn.Conv1d(conv_channels, 32, kernel_size=3, padding=1),
            actv_builder(),
            nn.Flatten(),
            nn.Linear(32 * seq_len, 1024),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class UnifiedQNetwork(nn.Module):
    def __init__(
        self,
        in_channels: int,
        action_space: int,
        seq_len: int,
        conv_channels: int = 256,
        num_blocks: int = 4,
    ):
        super().__init__()
        actv_builder = partial(nn.Mish, inplace=True)
        self.encoder = ResNet(
            in_channels=in_channels,
            conv_channels=conv_channels,
            num_blocks=num_blocks,
            seq_len=seq_len,
            actv_builder=actv_builder, # type: ignore
            pre_actv=True,
        )
        self.actv = actv_builder()
        self.fc_q = nn.Linear(1024, action_space)
        self._freeze_bn = False

    def forward(self, obs: Tensor) -> Tensor:
        phi = self.encoder(obs)
        phi = self.actv(phi)
        return self.fc_q(phi)

# ---------------------------------------------------------------------
# 2. Your exact configuration + test run
# ---------------------------------------------------------------------

if __name__ == "__main__":
    # Your exact instantiation, now with seq_len=29
    model = UnifiedQNetwork(
        in_channels=42,
        action_space=4,
        seq_len=52,         # <-- Set to 29!
        conv_channels=256,
        num_blocks=4,
    )

    # Your exact input shape
    obs = torch.randn(8, 42, 52)   # batch=8, channels=34, seq_len=29

    # Forward pass - no more shape errors!
    output = model(obs)
    print(f"Input shape:  {obs.shape}")
    print(f"Output shape: {output.shape}")  # Expected: torch.Size([8, 75])

    # Optional: count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

Input shape:  torch.Size([8, 42, 52])
Output shape: torch.Size([8, 4])
Total parameters: 3,377,252


In [4]:
class MultiAgentAdapter(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        if x.dim() == 3:
            # Input: (agents, C, L) – no batch dimension
            x = x.unsqueeze(0)                    # (1, agents, C, L)
            batch, agents, C, L = x.shape
            x_flat = x.view(batch * agents, C, L)
            out_flat = self.model(x_flat)        # (batch*agents, action_space)
            out = out_flat.view(batch, agents, -1)
            return out.squeeze(0)                # (agents, action_space)
        elif x.dim() == 4:
            # Input: (batch, agents, C, L)
            batch, agents, C, L = x.shape
            x_flat = x.view(batch * agents, C, L)
            out_flat = self.model(x_flat)
            out = out_flat.view(batch, agents, -1)
            return out
        else:
            raise ValueError(f"Expected 3D or 4D input, got {x.dim()}D")

class CastToFloat(nn.Module):
    def forward(self, x):
        return x.float()
# =====================================================================
# 3. Instantiate policy and critic with correct shapes
# =====================================================================

# ------------------- POLICY -------------------
base_net = UnifiedQNetwork(
    in_channels=34,
    action_space=75,
    seq_len=29,
    conv_channels=256,
    num_blocks=4,
)
policy_net = nn.Sequential(
    CastToFloat(),
    MultiAgentAdapter(base_net)
)
# Wrap with TensorDictModule
from tensordict.nn import TensorDictModule
policy_module = TensorDictModule(
    policy_net,
    in_keys=[("agents", "observation", "observation")],
    out_keys=[("agents", "logits")],
)

# Build the probabilistic actor
from torchrl.modules import ProbabilisticActor, MaskedCategorical
policy = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec_unbatched,
    in_keys={
        'logits': ('agents', 'logits'),
        'mask': ('agents', 'action_mask')
    }, # type: ignore
    out_keys=[env.action_key],
    distribution_class=MaskedCategorical,
    return_log_prob=True
)

Critic

In [5]:
class StateAdapter(nn.Module):
    def __init__(self, base_net, in_channels, agents=4):
        super().__init__()
        self.base_net = base_net
        self.in_channels = in_channels
        self.agents = agents

    def forward(self, x):
        added_batch = False
        if x.dim() == 2:                     # (C, L) -> add batch
            x = x.unsqueeze(0)               # (1, C, L)
            added_batch = True

        # Ensure correct channel order: (batch, channels, length)
        if x.shape[1] != self.in_channels:
            x = x.transpose(1, 2)            # (batch, L, C) -> (batch, C, L)

        out = self.base_net(x)               # (batch, agents)

        if added_batch:
            out = out.squeeze(0)             # (agents,)

        # Ensure last dimension is 1 (for state_value)
        if out.dim() == 1:
            out = out.unsqueeze(-1)          # (agents, 1)
        elif out.dim() == 2:
            out = out.unsqueeze(-1)          # (batch, agents, 1)
        return out
# ------------------- CRITIC -------------------
base_critic = UnifiedQNetwork(
    in_channels=42,
    action_space=4,
    seq_len=52,
    conv_channels=256,
    num_blocks=4,
)

critic_net = nn.Sequential(
    CastToFloat(),
    StateAdapter(base_critic, in_channels=42)
)
critic = TensorDictModule(
    module=critic_net,
    in_keys=["state"],
    out_keys=[("agents", "state_value")],
)

Initialize the model

In [37]:
policy = policy.to('cpu')
critic = critic.to('cpu')
x = policy(env.reset())
y = critic(env.reset())
actions = x[('agents', 'action')]
state_values = y[('agents', 'state_value')].squeeze()
print(f"Actions: {actions}, state_values: {state_values}")

Actions: tensor([32, 74,  6,  1]), state_values: tensor([ 0.1077, -0.0448,  0.0307,  0.0362], grad_fn=<SqueezeBackward0>)


Load models

In [ ]:
#

Rollout

In [ ]:
# import pygame
# from pygame_visualizer import render_game_state

# pygame.init()
# data = env.rollout(200, policy=policy.to('cpu'))
# screen = pygame.display.set_mode(size=(800, 800))
# font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)
# from sys import exit
# while True:
#     for event in pygame.event.get():
#         if event.type == pygame.QUIT:
#             pygame.quit()
#             exit()
#     screen.fill('white')
#     render_game_state(env._env.gamestate, screen, font)
#     pygame.display.update()

SystemExit: 

c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


Loss function and optimizer

In [ ]:
policy = policy.to(device)
loss_module = ClipPPOLoss(
    actor_network=policy, # type: ignore
    critic_network=critic,
    entropy_coeff=0.01
)
loss_module.set_keys(  # We have to tell the loss where to find the keys
    reward=env.reward_key,
    action=env.action_key,
    value=("agents", "state_value"),
    # These last 2 keys will be expanded to match the reward shape
    done=("agents", "done"),                # per-agent
    terminated=("agents", "terminated"),
)
gamma = 0.995  # discount factor
lmbda = 0.9  # lambda for generalised advantage estimation
lr = 5e-5
loss_module.make_value_estimator(
    ValueEstimators.GAE, gamma=gamma, lmbda=lmbda
)  
GAE = loss_module.value_estimator

optim = torch.optim.Adam(loss_module.parameters(), lr)

loss_module = loss_module.to(device)

Train loop

In [31]:
num_epochs = 5
max_grad_norm = 0.1
frames_per_batch = 1000  # Number of team frames collected per training iteration
n_iters = 10 # Number of sampling and training iterations
total_frames = frames_per_batch * n_iters
minibatch_size = 1000

if __name__ == "__main__":
    replay_buffer = ReplayBuffer(
        storage=LazyTensorStorage(
            frames_per_batch, device=device
        ),  # We store the frames_per_batch collected at each iteration
        sampler=SamplerWithoutReplacement(),
        batch_size=minibatch_size,  # We will sample minibatches of this siz
    )
    policy=policy.to(device)
    collector= Collector(
        env,
        policy=policy,
        device='cpu',
        storing_device=device,
        frames_per_batch=frames_per_batch,
        total_frames=total_frames
    )

    from tqdm.auto import tqdm
    for it, tensordict_data in enumerate(tqdm(collector)):
        tensordict_data.set(
            ("next", "agents", "done"),
            tensordict_data.get(("next", "done"))
            .unsqueeze(-1)
            .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
        )
        tensordict_data.set(
            ("next", "agents", "terminated"),
            tensordict_data.get(("next", "terminated"))
            .unsqueeze(-1)
            .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
        )
        # We need to expand the done and terminated to match the reward shape (this is expected by the value estimator)

        with torch.no_grad():
            GAE(
                tensordict_data,
                params=loss_module.critic_network_params,
                target_params=loss_module.target_critic_network_params,
            )  # Compute GAE and add it to the data

        data_view = tensordict_data.reshape(-1)  # Flatten the batch size to shuffle data
        replay_buffer.extend(data_view)

        for _ in range(num_epochs):
            for _ in range(frames_per_batch // minibatch_size):
                subdata = replay_buffer.sample()
                subdata = subdata.to(device)
                loss_vals = loss_module(subdata)

                loss_value = (
                    loss_vals["loss_objective"]
                    + loss_vals["loss_critic"]
                    + loss_vals["loss_entropy"]
                )

                loss_value.backward()

                torch.nn.utils.clip_grad_norm_(
                    loss_module.parameters(), max_grad_norm
                )  # Optional

                optim.step()
                optim.zero_grad()

        collector.update_policy_weights_()


C:\Users\ctc73\AppData\Local\Temp\ipykernel_15444\3068890342.py:17: FutureWarning: The env passed to Collector is missing transforms required by the policy (InitTracker). From torchrl v0.15 the collector will append them automatically. To enable that behavior now (and silence this warning), pass `auto_register_policy_transforms=True`. To opt out permanently, pass `auto_register_policy_transforms=False`.
  collector= Collector(
c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\tensordict\_td.py:612: FutureWarning: TensorDict.to_module() is replacing an existing nn.Parameter in the destination module with a tensor leaf that is not an nn.Parameter. This historical behavior can remove the key from module.state_dict(). In tensordict v0.14, to_module() will preserve existing module parameter and buffer registrations by default. Pass preserve_module_state=False to keep the current replacement behavior, or preserve_module_state=True to opt in to the v0.14 behavior now.
  local_o

2026-07-25 11:55:29,299 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([1000]) shape [END]


 90%|█████████ | 9/10 [13:14<01:34, 94.66s/it]

Yaku counted (with values):
  flowers (combined): 1
  tsumo: 1
  fan_pai: 1


100%|██████████| 10/10 [15:42<00:00, 94.24s/it] 


Rollout per step

In [11]:
import pygame
from sys import exit
from pygame_visualizer import render_game_state
from torchrl.envs.utils import step_mdp  # <-- Added missing utility

pygame.init()
screen = pygame.display.set_mode(size=(800, 800))
font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)

td = env.reset()
policy = policy.to('cpu')

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            exit()
            
        # Step through the game manually by pressing SPACE
        if event.type == pygame.KEYDOWN and event.key == pygame.K_SPACE:
            # 1. Check if the game is already over before stepping
            if td.get("done", torch.tensor([False])).any():
                print("Episode finished! Resetting environment...")
                td = env.reset()
                continue
                
            # 2. Policy reads root "observation" and writes root "action" into td
            td = policy(td)
            
            # 3. Environment executes the action and creates the "next" sub-TensorDict
            td = env.step(td)
            
            # 4. Check if this new step ended the game
            if td[("next", "done")].any():
                print(f"Game Over! Reward: {td[('next', 'agents', 'reward')]}")
            
            # 5. Move "next" keys to root level so the policy can read them next turn
            td = step_mdp(td)
            
    screen.fill('white')
    # Render the internal unwrapped environment game state
    render_game_state(env._env.gamestate, screen, font)
    pygame.display.update()

SystemExit: 

c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
